# Spatial Memory Demo: Ice & Water Navigation

**FLUX Phase 9 ARC** — Testing the dual-field spatial navigation system for ARC-AGI-3.

This notebook demonstrates:
1. **Exploration Mass Field** — tracks where agent has been
2. **Curiosity Ice Field** — highlights interesting locations
3. **Change Detection** — notices when visited areas change
4. **Path Planning** — efficient navigation via mass gradient

**Loads the REAL FLUX model from HuggingFace!**

## Cell 1: Setup & Environment Detection

In [ ]:
# ─────────────────────────────────────────────
# Setup for Kaggle / Google Colab / Local
# ─────────────────────────────────────────────
import sys
import os
from pathlib import Path

# Detect environment
IN_KAGGLE = os.path.exists('/kaggle')
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IN_LOCAL = not IN_KAGGLE and not IN_COLAB

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

# Clone repo if needed (Kaggle/Colab)
if IN_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
    if not ROOT.exists():
        print("Cloning FLUX repo...")
        os.system('git clone https://github.com/Unseengap/FLUX.git /kaggle/working/FLUX')
    # Install dependencies
    os.system('pip install -q huggingface_hub')
elif IN_COLAB:
    ROOT = Path('/content/FLUX')
    if not ROOT.exists():
        print("Cloning FLUX repo...")
        os.system('git clone https://github.com/Unseengap/FLUX.git /content/FLUX')
    # Install dependencies
    os.system('pip install -q torch matplotlib numpy huggingface_hub')
else:
    # Local — find project root
    ROOT = Path('.').resolve()
    while ROOT.name and not (ROOT / 'flux_utils.py').exists():
        ROOT = ROOT.parent
    if not (ROOT / 'flux_utils.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')  # Fallback

print(f"Project root: {ROOT}")

# Add paths
PHASES_DIR = ROOT / 'phases'
for p in [ROOT, PHASES_DIR / 'phase9_arc', PHASES_DIR / 'phase8_8',
          PHASES_DIR / 'phase8', PHASES_DIR / 'phase7', PHASES_DIR / 'phase6',
          PHASES_DIR / 'phase5', PHASES_DIR / 'phase4', PHASES_DIR / 'phase3',
          PHASES_DIR / 'phase2', PHASES_DIR / 'phase1']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

os.chdir(ROOT)
print(f"Working directory: {os.getcwd()}")

## Cell 2: Load HuggingFace Token from Secrets

In [ ]:
# ─────────────────────────────────────────────
# Get HuggingFace Token from Secrets
# ─────────────────────────────────────────────

HF_TOKEN = None

# Try Kaggle secrets first
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        HF_TOKEN = secrets.get_secret('HF_TOKEN')
        if HF_TOKEN:
            print("✓ HF_TOKEN loaded from Kaggle secrets")
    except Exception as e:
        print(f"⚠ Kaggle secrets error: {e}")

# Try Colab secrets
elif IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        if HF_TOKEN:
            print("✓ HF_TOKEN loaded from Colab secrets")
    except Exception as e:
        print(f"⚠ Colab secrets error: {e}")

# Try environment variable
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from environment")

# Try huggingface cached login
if not HF_TOKEN:
    try:
        from huggingface_hub import HfFolder
        HF_TOKEN = HfFolder.get_token()
        if HF_TOKEN:
            print("✓ HF_TOKEN loaded from cached login")
    except:
        pass

if HF_TOKEN:
    print(f"  Token: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")
else:
    print("⚠ No HF_TOKEN found! Model download may fail for private repos.")

## Cell 3: Download & Load FLUX Model

In [ ]:
%%time
# ─────────────────────────────────────────────
# Download Flux-X-complete.flx from HuggingFace
# ─────────────────────────────────────────────
import torch
from huggingface_hub import hf_hub_download

# Model settings
HF_REPO = "UnseenGAP/FLUX"
MODEL_FILE = "checkpoints/Flux-X-complete.flx"
LOCAL_PATH = ROOT / "checkpoints" / "Flux-X-complete.flx"

# Create checkpoints dir
LOCAL_PATH.parent.mkdir(parents=True, exist_ok=True)

# Download if not exists
if not LOCAL_PATH.exists():
    print(f"Downloading {MODEL_FILE} from HuggingFace...")
    downloaded = hf_hub_download(
        repo_id=HF_REPO,
        filename=MODEL_FILE,
        local_dir=str(ROOT),
        token=HF_TOKEN,
    )
    print(f"✓ Downloaded to {downloaded}")
else:
    print(f"✓ Model already exists: {LOCAL_PATH}")

# Check file size
size_mb = LOCAL_PATH.stat().st_size / 1e6
print(f"  Size: {size_mb:.1f} MB")

# Detect device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"  Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 4: Load FLUX Model Components

In [ ]:
%%time
# ─────────────────────────────────────────────
# Load the FLUX model from .flx archive
# ─────────────────────────────────────────────

print("Loading FLUX model...")

# Load raw .flx archive
flx = torch.load(str(LOCAL_PATH), map_location='cpu', weights_only=False)

print(f"✓ Loaded .flx archive")
print(f"  Format: {flx.get('format', 'unknown')}")
print(f"  Version: {flx.get('version', 'unknown')}")
print(f"  Components: {list(flx.keys())}")

# Load CSE (Continuous Semantic Encoder)
from cse import ContinuousSemanticEncoder
from wave_types import WAVE_DIMS

cse_config = flx['cse'].get('config', {})
cse = ContinuousSemanticEncoder(
    wave_dims=cse_config.get('wave_dims', dict(WAVE_DIMS)),
    byte_window=cse_config.get('byte_window', 8),
    byte_stride=cse_config.get('byte_stride', 1),
    interference_radius=cse_config.get('interference_radius', 4),
    device=device,
)
cse.load_state_dict(flx['cse']['state_dict'])
cse.to(device)
cse.eval()
print(f"✓ CSE loaded")

# Load Grid Adapters (for ARC)
from grid_adapters import GridToWave, WaveToGrid

grid_to_wave = GridToWave(wave_dim=432, device=device)
wave_to_grid = WaveToGrid(wave_dim=432, device=device)

# Try to load trained weights if available in .flx
if 'grid_adapters' in flx:
    if 'encoder' in flx['grid_adapters']:
        grid_to_wave.load_state_dict(flx['grid_adapters']['encoder'])
    if 'decoder' in flx['grid_adapters']:
        wave_to_grid.load_state_dict(flx['grid_adapters']['decoder'])
    print(f"✓ Grid adapters loaded from checkpoint")
else:
    print(f"✓ Grid adapters initialized (no trained weights in this checkpoint)")

grid_to_wave.to(device)
wave_to_grid.to(device)
grid_to_wave.eval()
wave_to_grid.eval()

print(f"\n✓ FLUX model ready on {device}!")

## Cell 5: Train GridToWave on Sample Grids

The GridToWave model needs training to produce discriminative embeddings.
We'll do a quick training on random ARC-style grids to get useful wave encodings.

In [ ]:
%%time
# ─────────────────────────────────────────────
# Train GridToWave for Discriminative Embeddings
# ─────────────────────────────────────────────
import torch.nn as nn
import torch.optim as optim
import random
from tqdm.auto import tqdm

print("Training GridToWave for discriminative embeddings...")
print("="*60)

# Training config
TRAIN_STEPS = 500
BATCH_SIZE = 16
LR = 1e-3
GRID_SIZE = 10
MARGIN = 0.3  # For triplet loss

# Put in training mode
grid_to_wave.train()

optimizer = optim.AdamW(grid_to_wave.parameters(), lr=LR)

def generate_random_grid(size=10, num_objects=4):
    """Generate random ARC-style grid."""
    grid = [[0] * size for _ in range(size)]
    for _ in range(num_objects):
        r, c = random.randint(0, size-1), random.randint(0, size-1)
        color = random.randint(1, 9)
        grid[r][c] = color
    return grid

def modify_grid(grid, num_changes=1):
    """Create a slightly modified version of the grid."""
    new_grid = [row[:] for row in grid]
    size = len(grid)
    for _ in range(num_changes):
        r, c = random.randint(0, size-1), random.randint(0, size-1)
        new_grid[r][c] = random.randint(0, 9)
    return new_grid

# Training loop with triplet-style loss
losses = []
pbar = tqdm(range(TRAIN_STEPS), desc="Training")

for step in pbar:
    optimizer.zero_grad()
    
    batch_loss = 0.0
    
    for _ in range(BATCH_SIZE // 2):
        # Anchor grid
        anchor_grid = generate_random_grid(GRID_SIZE)
        # Positive: same grid (should produce identical wave)
        positive_grid = anchor_grid
        # Negative: different grid
        negative_grid = generate_random_grid(GRID_SIZE)
        # Near-negative: slightly modified grid
        near_neg_grid = modify_grid(anchor_grid, num_changes=2)
        
        # Encode all
        anchor_wave = grid_to_wave.encode(anchor_grid, mode='holistic')
        positive_wave = grid_to_wave.encode(positive_grid, mode='holistic')
        negative_wave = grid_to_wave.encode(negative_grid, mode='holistic')
        near_neg_wave = grid_to_wave.encode(near_neg_grid, mode='holistic')
        
        # Loss 1: Anchor-Positive should be identical
        consistency_loss = (anchor_wave - positive_wave).pow(2).mean()
        
        # Loss 2: Triplet loss - anchor closer to positive than negative
        pos_dist = (anchor_wave - positive_wave).pow(2).sum().sqrt()
        neg_dist = (anchor_wave - negative_wave).pow(2).sum().sqrt()
        triplet_loss = torch.relu(pos_dist - neg_dist + MARGIN)
        
        # Loss 3: Near-negative should be somewhat different
        near_neg_dist = (anchor_wave - near_neg_wave).pow(2).sum().sqrt()
        # Want near_neg to have some distance (changed cells should show up)
        near_loss = torch.relu(0.1 - near_neg_dist)
        
        # Loss 4: Wave magnitude (prevent collapse to zero)
        magnitude_loss = torch.relu(5.0 - anchor_wave.norm())
        
        # Loss 5: Variance (prevent all waves being the same)
        all_waves = torch.stack([anchor_wave, negative_wave, near_neg_wave])
        variance_loss = torch.relu(0.5 - all_waves.var(dim=0).mean())
        
        batch_loss += consistency_loss + triplet_loss + near_loss + 0.1 * magnitude_loss + variance_loss
    
    batch_loss = batch_loss / (BATCH_SIZE // 2)
    batch_loss.backward()
    optimizer.step()
    
    losses.append(batch_loss.item())
    
    if step % 50 == 0:
        pbar.set_postfix({'loss': f'{batch_loss.item():.4f}'})

# Back to eval mode
grid_to_wave.eval()

print(f"\n✓ Training complete!")
print(f"  Final loss: {losses[-1]:.4f}")
print(f"  Initial loss: {losses[0]:.4f}")
print(f"  Improvement: {(1 - losses[-1]/losses[0])*100:.1f}%")

# Test discrimination
print("\nTesting wave discrimination...")
test_grid1 = [[0]*10 for _ in range(10)]
test_grid1[5][5] = 1  # Single red pixel

test_grid2 = [[0]*10 for _ in range(10)]
test_grid2[5][5] = 2  # Single teal pixel

test_grid3 = [[0]*10 for _ in range(10)]
test_grid3[2][2] = 1  # Same color, different position

with torch.no_grad():
    w1 = grid_to_wave.encode(test_grid1, mode='holistic')
    w2 = grid_to_wave.encode(test_grid2, mode='holistic')
    w3 = grid_to_wave.encode(test_grid3, mode='holistic')
    
    sim_12 = nn.functional.cosine_similarity(w1.unsqueeze(0), w2.unsqueeze(0)).item()
    sim_13 = nn.functional.cosine_similarity(w1.unsqueeze(0), w3.unsqueeze(0)).item()

print(f"  Red@5,5 vs Teal@5,5: {sim_12:.4f}")
print(f"  Red@5,5 vs Red@2,2:  {sim_13:.4f}")

if sim_12 < 0.95 and sim_13 < 0.95:
    print("✓ GridToWave produces discriminative embeddings!")
else:
    print("⚠ May need more training")

# Plot training curve
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(losses, 'b-', alpha=0.7)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('GridToWave Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## Cell 6: Load Spatial Memory Module

In [ ]:
# ─────────────────────────────────────────────
# Import Spatial Memory System
# ─────────────────────────────────────────────
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display, HTML

from spatial_memory import SpatialMemory, SpatialARCAgent, NavigationTarget

print("✓ Spatial Memory module loaded")
print(f"  PyTorch: {torch.__version__}")
print(f"  Device: {device}")

## Cell 7: Define ARC-AGI-3 Style Game

In [ ]:
class FLUXTreasureHuntEnv:
    """
    ARC-AGI-3 style exploration game powered by FLUX.
    """
    
    def __init__(self, size: int = 10, num_treasures: int = 4, num_hidden: int = 2):
        self.size = size
        self.num_treasures = num_treasures
        self.num_hidden = num_hidden
        self.reset()
    
    def reset(self):
        """Reset the environment."""
        self.agent_pos = [0, 0]
        self.steps = 0
        self.found_treasures = set()
        
        # Place visible treasures (colors 1-4)
        self.visible_treasures = {}
        positions = random.sample(
            [(r, c) for r in range(self.size) for c in range(self.size) if (r, c) != (0, 0)],
            k=self.num_treasures
        )
        for i, pos in enumerate(positions):
            self.visible_treasures[pos] = i + 1
        
        # Place hidden treasures (appear after trigger)
        self.hidden_treasures = {}
        self.revealed_hidden = set()
        
        trigger_positions = list(self.visible_treasures.keys())[:self.num_hidden]
        for i, trigger in enumerate(trigger_positions):
            available = [(r, c) for r in range(self.size) for c in range(self.size) 
                        if (r, c) not in self.visible_treasures and (r, c) != (0, 0)]
            if available:
                hidden_pos = random.choice(available)
                self.hidden_treasures[trigger] = (hidden_pos, 5 + i)
        
        return self.get_observation()
    
    def get_observation(self):
        """Get current grid state."""
        grid = [[0] * self.size for _ in range(self.size)]
        
        # Place visible treasures
        for pos, color in self.visible_treasures.items():
            if pos not in self.found_treasures:
                grid[pos[0]][pos[1]] = color
        
        # Place revealed hidden treasures
        for trigger, (pos, color) in self.hidden_treasures.items():
            if trigger in self.revealed_hidden and pos not in self.found_treasures:
                grid[pos[0]][pos[1]] = color
        
        return grid
    
    def get_observation_with_agent(self):
        """Get grid with agent marker."""
        grid = self.get_observation()
        grid[self.agent_pos[0]][self.agent_pos[1]] = 9  # Agent marker
        return grid
    
    def step(self, action: int):
        """Take action: 0=up, 1=down, 2=left, 3=right"""
        dr, dc = [(-1, 0), (1, 0), (0, -1), (0, 1)][action]
        new_r = max(0, min(self.size - 1, self.agent_pos[0] + dr))
        new_c = max(0, min(self.size - 1, self.agent_pos[1] + dc))
        self.agent_pos = [new_r, new_c]
        self.steps += 1
        
        pos = tuple(self.agent_pos)
        
        # Check treasure collection
        treasure_found = False
        hidden_revealed = False
        
        if pos in self.visible_treasures and pos not in self.found_treasures:
            self.found_treasures.add(pos)
            treasure_found = True
            if pos in self.hidden_treasures:
                self.revealed_hidden.add(pos)
                hidden_revealed = True
        
        for trigger, (hidden_pos, color) in self.hidden_treasures.items():
            if trigger in self.revealed_hidden and pos == hidden_pos and pos not in self.found_treasures:
                self.found_treasures.add(pos)
                treasure_found = True
        
        total = len(self.visible_treasures) + len(self.hidden_treasures)
        done = len(self.found_treasures) >= total
        
        return self.get_observation(), treasure_found, hidden_revealed, done
    
    def get_stats(self):
        total = len(self.visible_treasures) + len(self.hidden_treasures)
        return {
            'steps': self.steps,
            'found': len(self.found_treasures),
            'total': total,
            'revealed_hidden': len(self.revealed_hidden),
        }

print("✓ FLUXTreasureHuntEnv defined")

## Cell 8: FLUX-Powered Agent with Wave + Direct Grid Detection

In [ ]:
class FLUXSpatialAgent:
    """
    Agent that uses FLUX wave encoding + spatial memory + direct grid comparison.
    
    This agent:
    1. Encodes grid observations using GridToWave
    2. Uses BOTH wave similarity AND direct cell comparison
    3. Targets colored cells (non-zero) as curiosity points
    4. Uses spatial memory for navigation
    """
    
    def __init__(self, max_size: int = 30, verbose: bool = False):
        self.spatial_memory = SpatialMemory(max_size=max_size, device=device)
        self.current_pos = (0, 0)
        self.current_target = None
        self.action_history = []
        self.wave_history = []
        self.last_grid_wave = None
        self.last_grid_copy = None  # Store actual grid for direct comparison
        self.verbose = verbose
        self.logs = []
        self.treasure_positions_seen = set()
    
    def log(self, msg):
        """Add to logs, print if verbose."""
        self.logs.append(msg)
        if self.verbose:
            print(f"    [Agent] {msg}")
    
    def reset(self, initial_pos=(0, 0)):
        self.spatial_memory.reset()
        self.current_pos = initial_pos
        self.current_target = None
        self.action_history = []
        self.wave_history = []
        self.last_grid_wave = None
        self.last_grid_copy = None
        self.logs = []
        self.treasure_positions_seen = set()
        self.log(f"Reset at position {initial_pos}")
    
    def encode_grid_to_wave(self, grid):
        """Encode grid using FLUX GridToWave. Returns [432] wave tensor."""
        with torch.no_grad():
            wave = grid_to_wave.encode(grid, mode='holistic')
        return wave
    
    def detect_grid_changes(self, current_grid):
        """Direct grid comparison - find exact cell changes."""
        if self.last_grid_copy is None:
            return []
        
        changes = []
        for r in range(len(current_grid)):
            for c in range(len(current_grid[0])):
                old_val = self.last_grid_copy[r][c]
                new_val = current_grid[r][c]
                if old_val != new_val:
                    changes.append((r, c, old_val, new_val))
        return changes
    
    def find_treasure_cells(self, grid):
        """Find all non-zero cells (treasures/colored objects)."""
        treasures = []
        for r in range(len(grid)):
            for c in range(len(grid[0])):
                color = grid[r][c]
                if color > 0:
                    treasures.append((r, c, color))
        return treasures
    
    def detect_wave_change(self, current_wave):
        """Compare current wave to last wave. Returns change magnitude."""
        if self.last_grid_wave is None:
            return 0.0
        
        cos_sim = torch.nn.functional.cosine_similarity(
            current_wave.unsqueeze(0),
            self.last_grid_wave.unsqueeze(0)
        ).item()
        
        return 1.0 - cos_sim
    
    def observe_and_decide(self, grid, position):
        """
        Full observe → encode → plan → decide cycle.
        Uses BOTH wave encoding AND direct grid comparison.
        """
        self.current_pos = position
        grid_size = (len(grid), len(grid[0]))
        
        step_info = {'position': position, 'events': []}
        
        # 1. Encode grid to wave using FLUX
        current_wave = self.encode_grid_to_wave(grid)
        self.wave_history.append(current_wave.cpu())
        
        # 2. Wave-based change detection
        wave_change = self.detect_wave_change(current_wave)
        
        # 3. Direct grid change detection (fallback)
        grid_changes = self.detect_grid_changes(grid)
        direct_change = len(grid_changes) > 0
        
        if grid_changes:
            for r, c, old_val, new_val in grid_changes:
                self.log(f"Grid change at ({r},{c}): {old_val} → {new_val}")
                step_info['events'].append(f"change_at_{r}_{c}")
                # Spawn ice at change location
                self.spatial_memory.curiosity_field[r, c] += 5.0
        
        # 4. Find colored cells (treasures) and add curiosity
        treasures = self.find_treasure_cells(grid)
        for r, c, color in treasures:
            pos_key = (r, c)
            if pos_key not in self.treasure_positions_seen:
                self.treasure_positions_seen.add(pos_key)
                self.log(f"New treasure spotted at ({r},{c}) color={color}")
                step_info['events'].append(f"treasure_at_{r}_{c}")
                self.spatial_memory.curiosity_field[r, c] += 10.0
            else:
                if grid[r][c] != 0:  # Still there
                    self.spatial_memory.curiosity_field[r, c] += 1.0
        
        # 5. Update spatial memory
        obs_result = self.spatial_memory.observe(
            position=position,
            local_view=grid,
            global_grid=grid,
        )
        
        # 6. Boost curiosity for wave OR direct changes
        combined_change = max(wave_change, 1.0 if direct_change else 0.0)
        if combined_change > 0.01:
            r, c = position
            self.spatial_memory.curiosity_field[r, c] += combined_change * 5
        
        # 7. Detect anomalies
        anomalies = self.spatial_memory.detect_anomalies(grid)
        
        # 8. Decay fields
        self.spatial_memory.step_decay()
        
        # 9. Get navigation target
        needs_new_target = (
            self.current_target is None or
            self.current_pos == self.current_target.position or
            obs_result['change_detected'] or
            direct_change or
            wave_change > 0.01
        )
        
        if needs_new_target:
            self.current_target = self.spatial_memory.get_navigation_target(
                self.current_pos,
                grid_size,
            )
            if self.current_target:
                self.log(f"New target: {self.current_target.position} ({self.current_target.reason})")
        
        # 10. Decide action
        if self.current_target:
            action = self.spatial_memory.get_next_action(
                self.current_pos,
                self.current_target,
            )
        else:
            action = random.randint(0, 3)
            self.log("No target, random action")
        
        # Update state for next comparison
        self.last_grid_wave = current_wave
        self.last_grid_copy = [row[:] for row in grid]  # Deep copy
        
        self.action_history.append(action)
        
        return action, wave_change, direct_change, step_info
    
    def get_visualization(self, grid_size):
        return self.spatial_memory.visualize(grid_size, self.current_pos)
    
    def get_stats(self, grid_size):
        stats = self.spatial_memory.get_stats(grid_size)
        stats['actions_taken'] = len(self.action_history)
        stats['waves_encoded'] = len(self.wave_history)
        stats['treasures_seen'] = len(self.treasure_positions_seen)
        return stats
    
    def get_recent_logs(self, n=20):
        return self.logs[-n:]

print("✓ FLUXSpatialAgent defined with:")
print("  - Wave encoding (FLUX GridToWave)")
print("  - Direct grid comparison (fallback)")
print("  - Treasure detection (colored cells)")
print("  - Detailed logging")

## Cell 9: Visualization Helpers

In [ ]:
# Color mapping for visualization
COLORS = {
    0: '#1a1a2e',
    1: '#e63946',
    2: '#2a9d8f',
    3: '#e9c46a',
    4: '#9b59b6',
    5: '#00ff00',
    6: '#00ffff',
    7: '#ff8800',
    8: '#ff0088',
    9: '#ffffff',
}

def plot_game_state(env, agent, title="Game State", show_fields=True):
    """Plot the game state with field overlays."""
    
    if show_fields:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    else:
        fig, axes = plt.subplots(1, 1, figsize=(6, 6))
        axes = [axes]
    
    # Plot 1: Game grid
    grid = env.get_observation_with_agent()
    rgb_grid = np.zeros((env.size, env.size, 3))
    
    for r in range(env.size):
        for c in range(env.size):
            color = COLORS.get(grid[r][c], '#888888')
            rgb_grid[r, c] = [int(color[i:i+2], 16)/255 for i in (1, 3, 5)]
    
    axes[0].imshow(rgb_grid)
    axes[0].set_title(f"{title}\nStep: {env.steps}, Found: {len(env.found_treasures)}")
    axes[0].set_xticks(range(env.size))
    axes[0].set_yticks(range(env.size))
    axes[0].grid(True, color='gray', linewidth=0.5)
    
    if show_fields and len(axes) > 1:
        mass = agent.spatial_memory.exploration_mass[:env.size, :env.size].cpu().numpy()
        im = axes[1].imshow(mass, cmap='hot', vmin=0)
        axes[1].set_title("Exploration Mass\n(Where I've been)")
        plt.colorbar(im, ax=axes[1])
        
        ice = agent.spatial_memory.curiosity_field[:env.size, :env.size].cpu().numpy()
        im = axes[2].imshow(ice, cmap='cool', vmin=0)
        axes[2].set_title("Curiosity Ice\n(What's interesting)")
        plt.colorbar(im, ax=axes[2])
    
    plt.tight_layout()
    plt.show()

print("✓ Visualization helpers defined")

## Cell 10: Initialize Game and Agent

In [ ]:
%%time
# ─────────────────────────────────────────────
# Run FLUX-Powered Treasure Hunt
# ─────────────────────────────────────────────

# Create game and FLUX-powered agent
env = FLUXTreasureHuntEnv(size=10, num_treasures=4, num_hidden=2)
agent = FLUXSpatialAgent(max_size=10, verbose=True)  # Verbose for detailed logs

# Reset
grid = env.reset()
agent.reset(initial_pos=(0, 0))

print("="*60)
print("FLUX-POWERED TREASURE HUNT")
print("="*60)
print(f"Grid size: {env.size}x{env.size}")
print(f"Visible treasures: {len(env.visible_treasures)}")
print(f"Hidden treasures: {len(env.hidden_treasures)}")
print(f"Total to find: {len(env.visible_treasures) + len(env.hidden_treasures)}")
print()
print("Treasure locations:")
for pos, color in env.visible_treasures.items():
    is_trigger = pos in env.hidden_treasures
    trigger_info = f" → triggers hidden at {env.hidden_treasures[pos][0]}" if is_trigger else ""
    print(f"  {pos}: color={color}{trigger_info}")
print()
print(f"Using FLUX GridToWave for observation encoding!")
print("="*60)

# Initial state
plot_game_state(env, agent, "Initial State")

## Cell 11: Run Game Loop with Detailed Logging

In [ ]:
%%time
# ─────────────────────────────────────────────
# Run the game with detailed logging
# ─────────────────────────────────────────────
MAX_STEPS = 200
PRINT_EVERY = 25

print("Running FLUX-powered exploration...")
print("="*60)

done = False
step = 0
events = []
wave_changes = []
direct_changes = []

while not done and step < MAX_STEPS:
    pos = tuple(env.agent_pos)
    
    # Agent observes and decides
    action, wave_change, direct_change, step_info = agent.observe_and_decide(grid, pos)
    wave_changes.append(wave_change)
    direct_changes.append(1 if direct_change else 0)
    
    # Take step in environment
    old_pos = pos
    grid, treasure_found, hidden_revealed, done = env.step(action)
    new_pos = tuple(env.agent_pos)
    step += 1
    
    # Log events
    action_name = ['↑up', '↓down', '←left', '→right'][action]
    
    if treasure_found:
        events.append(f"Step {step}: 💎 COLLECTED at {old_pos}")
        print(f"  Step {step}: 💎 TREASURE at {old_pos}!")
    
    if hidden_revealed:
        hidden_pos = env.hidden_treasures[old_pos][0]
        events.append(f"Step {step}: 🔮 HIDDEN revealed at {hidden_pos}")
        print(f"  Step {step}: 🔮 HIDDEN revealed at {hidden_pos}!")
    
    if direct_change:
        events.append(f"Step {step}: 📍 Grid change detected")
    
    if wave_change > 0.01:
        events.append(f"Step {step}: ⚡ Wave Δ={wave_change:.4f}")
    
    # Periodic detailed update
    if step % PRINT_EVERY == 0 or done:
        stats = env.get_stats()
        agent_stats = agent.get_stats((env.size, env.size))
        avg_wave = sum(wave_changes[-25:]) / min(25, len(wave_changes))
        direct_rate = sum(direct_changes[-25:]) / min(25, len(direct_changes))
        
        print(f"\n─── Step {step} ───")
        print(f"  Position: {new_pos}")
        print(f"  Found: {stats['found']}/{stats['total']} treasures")
        print(f"  Hidden revealed: {stats['revealed_hidden']}/{len(env.hidden_treasures)}")
        print(f"  Coverage: {agent_stats['coverage']:.1%}")
        print(f"  Treasures seen: {agent_stats['treasures_seen']}")
        print(f"  Avg wave Δ: {avg_wave:.4f}")
        print(f"  Direct change rate: {direct_rate:.1%}")
        
        if agent.current_target:
            print(f"  Current target: {agent.current_target.position} ({agent.current_target.reason})")
        
        # Show ice peaks
        ice_field = agent.spatial_memory.curiosity_field[:env.size, :env.size]
        top_ice = torch.topk(ice_field.flatten(), k=3)
        print(f"  Top curiosity:", end=" ")
        for idx, val in zip(top_ice.indices, top_ice.values):
            r, c = idx.item() // env.size, idx.item() % env.size
            print(f"({r},{c})={val.item():.1f}", end=" ")
        print()

# ─────────────────────────────────────────────
# Final Results
# ─────────────────────────────────────────────
print()
print("="*60)
print("GAME OVER")
print("="*60)

stats = env.get_stats()
agent_stats = agent.get_stats((env.size, env.size))

print(f"\n📊 FINAL STATS:")
print(f"  Steps taken: {stats['steps']}")
print(f"  Treasures found: {stats['found']}/{stats['total']}")
print(f"  Hidden revealed: {stats['revealed_hidden']}/{len(env.hidden_treasures)}")
print(f"  Grid coverage: {agent_stats['coverage']:.1%}")
print(f"  Waves encoded: {agent_stats['waves_encoded']}")
print(f"  Treasures spotted: {agent_stats['treasures_seen']}")
print(f"  Result: {'✓ SUCCESS' if done else '✗ FAILED'}")

print(f"\n📜 KEY EVENTS ({len(events)} total):")
for event in events[-20:]:
    print(f"  {event}")

print(f"\n📝 AGENT LOGS (last 15):")
for log in agent.get_recent_logs(15):
    print(f"  {log}")

## Cell 12: Visualize Final State

In [ ]:
# Final state visualization
plot_game_state(env, agent, "Final State", show_fields=True)

# Plot wave changes over time
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(wave_changes, 'b-', alpha=0.7, label='Wave Change')
axes[0].axhline(y=0.01, color='r', linestyle='--', label='Change Threshold')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Wave Change (1 - cosine sim)')
axes[0].set_title('FLUX Wave Change Detection')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(direct_changes, 'g-', alpha=0.7, label='Direct Change', drawstyle='steps')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Grid Changed (0/1)')
axes[1].set_title('Direct Grid Change Detection')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ASCII visualization
print("\nSpatial Memory State:")
print(agent.get_visualization((env.size, env.size)))

## Summary

This demo showed the **FLUX-powered spatial memory system**:

| Component | How It Works |
|-----------|-------------|
| **GridToWave** | Encodes ARC grids → 432D wave vectors |
| **Wave Change** | Compares wave similarity to detect changes |
| **Direct Grid** | Cell-by-cell comparison as fallback |
| **Exploration Mass** | Tracks visited locations via gravitational breadcrumbs |
| **Curiosity Ice** | Highlights anomalies and change sites |
| **Navigation** | A* pathfinding with mass gradient bonus |

The agent:
- Uses **FLUX wave encoding** for observations
- Detects changes via **wave + direct grid** comparison
- Navigates using **spatial memory fields**

In [ ]:
print("="*60)
print("✓ FLUX Spatial Memory Demo Complete")
print("="*60)
print()
print(f"Model used: Flux-X-complete.flx")
print(f"Device: {device}")
print(f"Final stats:")
print(f"  - Steps: {stats['steps']}")
print(f"  - Success: {done}")
print(f"  - Coverage: {agent_stats['coverage']:.1%}")
print(f"  - Treasures: {stats['found']}/{stats['total']}")
print()
print("Next steps:")
print("  1. Test on real ARC-AGI-3 environments")
print("  2. Tune curiosity/exploration parameters")
print("  3. Add causal tracking (why ice formed)")